# Classical Image Processing

Notebook for **Chapter 8** of the *dl* study document. Run the cells top to bottom. Read the full explanation on the site: https://dadipavan.github.io/Learning/study/dl/ch08.html

Tip: change a number, predict the output, then run the cell again.

In [ ]:
import numpy as np
def show(a, chars=" .:-=+*#%@"):
    """print a 2D array as ASCII art (each pixel is 2 characters wide)"""
    a = np.asarray(a, float); lo, hi = a.min(), a.max()
    q = ((a - lo) / (hi - lo + 1e-9) * (len(chars) - 1)).round().astype(int)
    print("\n".join("".join(chars[v] * 2 for v in row) for row in q))

def conv2d(x, k):
    """same-size 2D filtering (cross-correlation), edge pixels repeated"""
    kh, kw = k.shape; ph, pw = kh // 2, kw // 2
    xp = np.pad(x, ((ph, ph), (pw, pw)), mode='edge')
    out = np.zeros(x.shape, float)
    for i in range(x.shape[0]):
        for j in range(x.shape[1]):
            out[i, j] = (xp[i:i+kh, j:j+kw] * k).sum()
    return out

## 8.1 Pixels, Color Spaces, Resizing and Histogram Equalization

In [ ]:
import colorsys
px = np.array([200, 100, 50.])
print("gray", round(float(px @ np.array([0.299, 0.587, 0.114])), 2))
print("hsv", [round(float(v), 4) for v in colorsys.rgb_to_hsv(*(px / 255))])

a = np.array([[10., 20.], [30., 40.]])
def bilinear(a, H, W):
    h, w = a.shape; out = np.zeros((H, W))
    for i in range(H):
        for j in range(W):
            y = i * (h - 1) / (H - 1); x = j * (w - 1) / (W - 1)
            y0, x0 = int(np.floor(y)), int(np.floor(x))
            y1, x1 = min(y0 + 1, h - 1), min(x0 + 1, w - 1)
            fy, fx = y - y0, x - x0
            out[i, j] = (a[y0,x0]*(1-fy)*(1-fx) + a[y0,x1]*(1-fy)*fx
                         + a[y1,x0]*fy*(1-fx) + a[y1,x1]*fy*fx)
    return out
print(bilinear(a, 3, 3))
print(a[np.ix_([0, 0, 1], [0, 0, 1])])      # nearest-neighbour 2x2 -> 3x3 (rows/cols 0,0,1)

m = np.arange(6).reshape(2, 3)
print(np.rot90(m)); print(m[::-1])

gray 124.2
hsv [0.0556, 0.75, 0.7843]
[[10. 15. 20.]
 [20. 25. 30.]
 [30. 35. 40.]]
[[10. 10. 20.]
 [10. 10. 20.]
 [30. 30. 40.]]
[[2 5]
 [1 4]
 [0 3]]
[[3 4 5]
 [0 1 2]]


In [ ]:
im = np.array([[1,1,2,2],[1,2,2,3],[2,2,3,3],[2,3,3,4]])   # 4x4 image, gray levels 0..7
cnt = np.bincount(im.ravel(), minlength=8); cdf = cnt.cumsum()
print("counts", cnt); print("cdf   ", cdf)
cmin = cdf[cdf > 0].min(); n = im.size
lut = np.round((cdf - cmin) / (n - cmin) * 7).astype(int); lut[cdf == 0] = 0
print("lut   ", lut)
print(lut[im])

counts [0 3 7 5 1 0 0 0]
cdf    [ 0  3 10 15 16 16 16 16]
lut    [0 0 4 6 7 7 7 7]
[[0 0 4 4]
 [0 4 4 6]
 [4 4 6 6]
 [4 6 6 7]]


## 8.2 Filtering: Blur, Median and Sharpen

In [ ]:
patch = np.array([[10,20,30],[40,250,60],[70,80,90]], float)   # 250 is a bright noise spike
mean_k = np.ones((3,3)) / 9
gauss_k = np.array([[1,2,1],[2,4,2],[1,2,1]]) / 16
sharp_k = np.array([[0,-1,0],[-1,5,-1],[0,-1,0]], float)
print("mean  ", round(float((patch * mean_k).sum()), 2))
print("gauss ", float((patch * gauss_k).sum()))
print("median", float(np.median(patch)))
print("sharp on flat 10 with centre 50:", float((np.array([[10,10,10],[10,50,10],[10,10,10]]) * sharp_k).sum()))

mean   72.22
gauss  100.0
median 60.0
sharp on flat 10 with centre 50: 210.0


In [ ]:
rng = np.random.default_rng(1)
img = np.full((16, 16), 40.0); img[4:12, 4:12] = 200          # clean: dark floor, bright square
gaussian_noisy = np.clip(img + rng.normal(0, 25, img.shape), 0, 255)
g = np.exp(-(np.arange(5) - 2) ** 2 / (2 * 1.0 ** 2)); G = np.outer(g, g); G /= G.sum()   # 5x5 Gaussian, sigma 1
print(G.round(4)); print("sum of weights", round(float(G.sum()), 4))
def median(x, k=3):
    p = k // 2; xp = np.pad(x, p, mode='edge'); out = np.zeros_like(x)
    for i in range(x.shape[0]):
        for j in range(x.shape[1]): out[i, j] = np.median(xp[i:i+k, j:j+k])
    return out
sp = img.copy(); idx = rng.choice(256, 20, replace=False)
sp.flat[idx] = 255 * rng.integers(0, 2, 20)                   # 20 salt-and-pepper pixels
err = lambda a: round(float(np.abs(a - img).mean()), 2)       # mean absolute error vs clean
print("gaussian noise : noisy", err(gaussian_noisy), "| gauss blur", err(conv2d(gaussian_noisy, G)), "| median", err(median(gaussian_noisy)))
print("salt & pepper  : noisy", err(sp), "| gauss blur", err(conv2d(sp, G)), "| median", err(median(sp)))

[[0.003  0.0133 0.0219 0.0133 0.003 ]
 [0.0133 0.0596 0.0983 0.0596 0.0133]
 [0.0219 0.0983 0.1621 0.0983 0.0219]
 [0.0133 0.0596 0.0983 0.0596 0.0133]
 [0.003  0.0133 0.0219 0.0133 0.003 ]]
sum of weights 1.0
gaussian noise : noisy 17.26 | gauss blur 15.61 | median 10.78
salt & pepper  : noisy 12.01 | gauss blur 21.92 | median 2.5


## 8.3 Edge Detection: Sobel, Canny and Hough

In [ ]:
patch = np.array([[10,10,50,50,50]] * 5, float)               # dark left, bright right
Kx = np.array([[-1,0,1],[-2,0,2],[-1,0,1]], float); Ky = Kx.T
def valid(x, k):
    kh, kw = k.shape
    return np.array([[(x[i:i+kh, j:j+kw] * k).sum() for j in range(x.shape[1]-kw+1)]
                     for i in range(x.shape[0]-kh+1)])
gx, gy = valid(patch, Kx), valid(patch, Ky)
print("gx\n", gx); print("gy\n", gy)
print("magnitude row 0:", np.hypot(gx, gy)[0], " angle row 0:", np.degrees(np.arctan2(gy, gx))[0])

gx
 [[160. 160.   0.]
 [160. 160.   0.]
 [160. 160.   0.]]
gy
 [[0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]]
magnitude row 0: [160. 160.   0.]  angle row 0: [0. 0. 0.]


In [ ]:
Kx = np.array([[-1,0,1],[-2,0,2],[-1,0,1]], float); Ky = Kx.T
rng = np.random.default_rng(1)
img = np.full((16, 16), 40.0); img[4:12, 4:12] = 200
noisy = np.clip(img + rng.normal(0, 50, img.shape), 0, 255)     # strong noise
g = np.exp(-(np.arange(5) - 2) ** 2 / 2.0); G = np.outer(g, g); G /= G.sum()
mag = lambda x: np.hypot(conv2d(x, Kx), conv2d(x, Ky))
true_edge = mag(img) > 300
for name, x in [("no blur", noisy), ("blur first", conv2d(noisy, G))]:
    pred = mag(x) > 300
    print(f"{name:10s} flagged {pred.sum():3d} | true edge pixels {true_edge.sum()} | hit {(pred & true_edge).sum()} | false alarms {(pred & ~true_edge).sum()}")
show(mag(conv2d(noisy, G)) > 300, ".#")

no blur    flagged  79 | true edge pixels 60 | hit 58 | false alarms 21
blur first flagged  56 | true edge pixels 60 | hit 56 | false alarms 0
................................
................................
................................
........################........
........##################......
......####............####......
......####............####......
......####..............##......
......####..............##......
......####............####......
......####............####......
......####################......
........##############..........
................................
................................
................................


In [ ]:
im = np.zeros((10, 10))
for i in range(10): im[i, i] = 1                              # one diagonal line y = x
ys, xs = np.where(im > 0)
for deg in (0, 45, 90, 135):
    t = np.deg2rad(deg); rho = np.round(xs * np.cos(t) + ys * np.sin(t), 6)
    vals, c = np.unique(rho, return_counts=True)
    print("angle", deg, "most votes for one rho:", int(c.max()), "(rho =", float(vals[c.argmax()]), ")")

angle 0 most votes for one rho: 1 (rho = 0.0 )
angle 45 most votes for one rho: 1 (rho = 0.0 )
angle 90 most votes for one rho: 1 (rho = 0.0 )
angle 135 most votes for one rho: 10 (rho = 0.0 )


## 8.4 Thresholding, Morphology and Connected Components

In [ ]:
cnt = np.array([8, 12, 5, 3, 10, 14.]); lv = np.arange(6); tot = cnt.sum()
for t in range(5):                                            # class 0 = levels <= t
    w0 = cnt[:t+1].sum() / tot; w1 = 1 - w0
    m0 = (cnt[:t+1] * lv[:t+1]).sum() / cnt[:t+1].sum()
    m1 = (cnt[t+1:] * lv[t+1:]).sum() / cnt[t+1:].sum()
    print(f"t={t} w0={w0:.3f} mean0={m0:.3f} mean1={m1:.3f} between-class variance={w0*w1*(m0-m1)**2:.3f}")

t=0 w0=0.154 mean0=0.000 mean1=3.205 between-class variance=1.337
t=1 w0=0.385 mean0=0.600 mean1=4.031 between-class variance=2.787
t=2 w0=0.481 mean0=0.880 mean1=4.407 between-class variance=3.106
t=3 w0=0.538 mean0=1.107 mean1=4.583 between-class variance=3.003
t=4 w0=0.731 mean0=1.868 mean1=5.000 between-class variance=1.929


In [ ]:
def otsu(x):
    p = np.bincount(x.ravel().astype(int), minlength=256) / x.size
    best, bt = -1, 0
    for t in range(255):
        w0 = p[:t+1].sum(); w1 = 1 - w0
        if w0 == 0 or w1 == 0: continue
        m0 = (p[:t+1] * np.arange(t+1)).sum() / w0
        m1 = (p[t+1:] * np.arange(t+1, 256)).sum() / w1
        s = w0 * w1 * (m0 - m1) ** 2
        if s > best: best, bt = s, t
    return bt
def adaptive(x, k=9, c=5):                                    # pixel > local mean + c
    p = k // 2; xp = np.pad(x, p, mode='edge'); out = np.zeros(x.shape, bool)
    for i in range(x.shape[0]):
        for j in range(x.shape[1]): out[i, j] = x[i, j] > xp[i:i+k, j:j+k].mean() + c
    return out
rng = np.random.default_rng(2)
img = np.full((20, 20), 60.0); img[5:15, 5:15] = 180          # dark paper, bright object
truth = img > 100
even = np.clip(img + rng.normal(0, 20, img.shape), 0, 255)
t = otsu(even); print("even light  : otsu threshold", t, "accuracy", ((even > t) == truth).mean())
ramp = np.linspace(0, 120, 20)[None, :] * np.ones((20, 1))   # lighting gets brighter to the right
uneven = np.clip(img + ramp + rng.normal(0, 10, img.shape), 0, 255)
t2 = otsu(uneven); print("uneven light: otsu threshold", t2, "accuracy", round(float(((uneven > t2) == truth).mean()), 3))
print("uneven light: adaptive accuracy", round(float((adaptive(uneven) == truth).mean()), 3))

even light  : otsu threshold 111 accuracy 1.0
uneven light: otsu threshold 151 accuracy 0.777
uneven light: adaptive accuracy 0.902


In [ ]:
def erode(b, k=3):
    p = k // 2; bp = np.pad(b, p); out = np.zeros_like(b)
    for i in range(b.shape[0]):
        for j in range(b.shape[1]): out[i, j] = bp[i:i+k, j:j+k].all()
    return out
def dilate(b, k=3):
    p = k // 2; bp = np.pad(b, p); out = np.zeros_like(b)
    for i in range(b.shape[0]):
        for j in range(b.shape[1]): out[i, j] = bp[i:i+k, j:j+k].any()
    return out
b = np.zeros((10, 10), bool)
b[2:8, 2:8] = True; b[5, 5] = False          # a square with a one-pixel hole
b[0, 9] = True; b[9, 0] = True               # two speckles of noise
print("input"); show(b, ".#")
print("opening = erode then dilate (hole grows, speckles vanish)"); show(dilate(erode(b)), ".#")
closed = erode(dilate(b))                    # closing = dilate then erode
print("closing (hole filled, speckles stay)"); show(closed, ".#")
print("closing then opening (clean)"); show(dilate(erode(closed)), ".#")

input
..................##
....................
....############....
....############....
....############....
....######..####....
....############....
....############....
....................
##..................
opening = erode then dilate (hole grows, speckles vanish)
....................
....................
....############....
....############....
....############....
....######..........
....######..........
....######..........
....................
....................
closing (hole filled, speckles stay)
....................
....................
....############....
....############....
....############....
....############....
....############....
....############....
....................
....................
closing then opening (clean)
....................
....................
....############....
....############....
....############....
....############....
....############....
....############....
....................
....................


In [ ]:
def label(b):
    lab = np.zeros(b.shape, int); n = 0
    for i in range(b.shape[0]):
        for j in range(b.shape[1]):
            if b[i, j] and lab[i, j] == 0:
                n += 1; stack = [(i, j)]; lab[i, j] = n
                while stack:
                    y, x = stack.pop()
                    for dy, dx in ((1,0), (-1,0), (0,1), (0,-1)):
                        yy, xx = y + dy, x + dx
                        if 0 <= yy < b.shape[0] and 0 <= xx < b.shape[1] and b[yy, xx] and lab[yy, xx] == 0:
                            lab[yy, xx] = n; stack.append((yy, xx))
    return lab, n
b = np.zeros((10, 10), bool); b[2:8, 2:8] = True; b[5, 5] = False; b[0, 9] = True; b[9, 0] = True
lab, n = label(b); print("components:", n)
for k in range(1, n + 1):
    ys, xs = np.where(lab == k)
    print(k, "area", len(ys), "bbox (x0,y0,x1,y1)", tuple(int(v) for v in (xs.min(), ys.min(), xs.max(), ys.max())))

components: 3
1 area 1 bbox (x0,y0,x1,y1) (9, 0, 9, 0)
2 area 35 bbox (x0,y0,x1,y1) (2, 2, 7, 7)
3 area 1 bbox (x0,y0,x1,y1) (0, 9, 0, 9)


## 8.5 Features, Homography, Stitching and the Fourier View

In [ ]:
img = np.zeros((14, 14)); img[4:10, 4:10] = 1.0               # white square on black
Kx = np.array([[-1,0,1],[-2,0,2],[-1,0,1]], float) / 8; Ky = Kx.T
Ix, Iy = conv2d(img, Kx), conv2d(img, Ky); box = np.ones((3, 3))
Sxx, Syy, Sxy = conv2d(Ix*Ix, box), conv2d(Iy*Iy, box), conv2d(Ix*Iy, box)
k = 0.04; R = (Sxx*Syy - Sxy**2) - k * (Sxx + Syy) ** 2
print("R max", R.max().round(4), "R min", R.min().round(4))
print("corner pixels (R > 0.5*max):", np.argwhere(R > 0.5 * R.max()).tolist())
for name, (i, j) in [("corner", (4, 4)), ("edge", (4, 7)), ("flat", (7, 7))]:
    M = np.array([[Sxx[i,j], Sxy[i,j]], [Sxy[i,j], Syy[i,j]]])
    print(f"{name:6s} at {(i, j)} eigenvalues {np.linalg.eigvalsh(M).round(4)} R {R[i,j]:.4f}")

R max 0.492 R min -0.09
corner pixels (R > 0.5*max): [[4, 4], [4, 5], [4, 8], [4, 9], [5, 4], [5, 5], [5, 8], [5, 9], [8, 4], [8, 5], [8, 8], [8, 9], [9, 4], [9, 5], [9, 8], [9, 9]]
corner at (4, 4) eigenvalues [0.5625 1.0625] R 0.4920
edge   at (4, 7) eigenvalues [0.  1.5] R -0.0900
flat   at (7, 7) eigenvalues [0. 0.] R 0.0000


In [ ]:
src = np.array([[0,0],[1,0],[1,1],[0,1]], float)              # a unit square (the "document")
dst = np.array([[10,10],[50,20],[60,70],[5,60]], float)       # where its corners appear in the photo
A = []
for (x, y), (u, v) in zip(src, dst):
    A.append([-x, -y, -1, 0, 0, 0, u*x, u*y, u])
    A.append([0, 0, 0, -x, -y, -1, v*x, v*y, v])
_, _, Vt = np.linalg.svd(np.array(A)); H = Vt[-1].reshape(3, 3); H /= H[2, 2]
print(H.round(3))
for x, y in [(0, 0), (1, 1), (0.5, 0.5)]:
    q = H @ np.array([x, y, 1.0]); print((x, y), "->", (q[:2] / q[2]).round(2))

[[42.83  -6.415 10.   ]
 [11.132 33.019 10.   ]
 [ 0.057 -0.283  1.   ]]
(0, 0) -> [10. 10.]
(1, 1) -> [60. 70.]
(0.5, 0.5) -> [31.81 36.17]


In [ ]:
n = 32; t = np.arange(n)
x = np.sin(2*np.pi*2*t/n) + 0.5*np.sin(2*np.pi*10*t/n)        # slow wave (2 cycles) + fast wave (10 cycles)
mag = np.abs(np.fft.fft(x))[:n//2]
print("magnitude at bins 0,2,10:", mag[[0, 2, 10]].round(2), " at bins 1,3,5:", mag[[1, 3, 5]].round(2))
rng = np.random.default_rng(1); im = np.zeros((16, 16)); im[4:12, 4:12] = 1
noisy = im + rng.normal(0, 0.3, im.shape)
F = np.fft.fftshift(np.fft.fft2(noisy)); yy, xx = np.mgrid[:16, :16]
mask = ((yy - 8)**2 + (xx - 8)**2) <= 4**2                    # keep only low frequencies near the centre
den = np.fft.ifft2(np.fft.ifftshift(F * mask)).real
print("mean error: noisy", np.abs(noisy - im).mean().round(3), "| after low-pass", np.abs(den - im).mean().round(3))

magnitude at bins 0,2,10: [ 0. 16.  8.]  at bins 1,3,5: [0. 0. 0.]
mean error: noisy 0.213 | after low-pass 0.145


## Chapter recap

## Practice set

**1.** (easy) Grey value of `(100, 150, 200)` with luma weights. **Answer:** `29.9 + 88.05 + 22.8 = 140.75`.

## Prompt pack

---
Done? Do the **Practice set** at the end of the chapter, then clear your Review cards on the site.

https://dadipavan.github.io/Learning/study/dl/ch08.html